# Modelisation du jeu de données nettoyé

Ce notebook présente une exploration PCA en 3D, un découpage train/test/validation, un appel à une fonction de prétraitement externe, et un entraînement en boucle de plusieurs modèles. *Ce workflow doit avoir le jeux de données nettoyé en entrée*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re

# add root path, to import all modules
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    
from src.accidents.data import load_collisions
from src.accidents.models.clustering import plot_3d, compute_methods

from sklearn.decomposition import PCA
from sklearn.manifold import MDS, Isomap
from sklearn.model_selection import train_test_split
import plotly.express as px

seed = 3795

DATA_PATH = Path('../data/clean/collisions_clean.csv')
df = load_collisions(DATA_PATH)

### Choisir la colonne *target*

In [18]:
target = 'GRAVITE_3'
# target = 'GRAVITE'

classes = df[target].unique()
c = len(classes)
print(classes)

<StringArray>
['Materiel', 'Leger', 'Grave']
Length: 3, dtype: str


### Loader les colonnes et split

In [ ]:
gravite_column = next((c for c in df.columns if c.upper() == target), None)
if gravite_column is None:
    raise ValueError(f'Impossible de trouver une colonne {target} dans le jeu de données nettoyé.')

# X = toutes les colonnes sauf la cible
# Les colonnes de fuite (NB_MORTS, NB_BLESSES_*, etc.) seront retirees par preprocess_data()
feature_columns = [c for c in df.columns if c.upper() not in {'GRAVITE', 'GRAVITE_3'}]
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if not feature_columns:
    raise ValueError(f'Aucune colonne numérique disponible pour la PCA après exclusion de {target}.')

X = df[feature_columns].copy()
y = df[gravite_column].copy()

print(f'Features disponibles : {len(feature_columns)}')
print(f'Distribution de la cible :')
print(y.value_counts())

## Découpage train / validation / test

Nous préparons le jeu de données pour l'entraînement en séparant d'abord un test set, puis en divisant le reste entre entraînement et validation.

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=seed
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=seed
)

print('Tailles :')
print('  X_train', X_train.shape)
print('  X_val', X_val.shape)
print('  X_test', X_test.shape)

### Check des petites classes

In [6]:
print("Proportions de chaque classe des ensembles:")
print('  Y_train_proc', y_train.value_counts())
print('  Y_val_proc', y_val.value_counts())
print('  Y_test_proc', y_test.value_counts())

## Appel au prétraitement externe

Ce notebook invoque la fonction `preprocess_data` située dans `src.accidents.preprocessing`. Cette fonction applique :
1. Le retrait des colonnes qui fuitent la cible (`NB_MORTS`, `NB_BLESSES_*`, etc.)
2. Un target encoding sur les catégorielles à forte cardinalité (`REG_ADM`, `MRC`)
3. Un one-hot encoding sur les autres catégorielles
4. Une standardisation (z-score) de toutes les colonnes

Toutes les statistiques sont calculées uniquement sur `X_train` pour éviter toute fuite.

In [7]:
from src.accidents.preprocessing.preprocessing import preprocess_data

X_train_proc, X_val_proc, X_test_proc = preprocess_data(X_train, y_train, X_val, X_test)

print('Prétraitement terminé. Formes finales :')
print('  X_train_proc', X_train_proc.shape)
print('  X_val_proc', X_val_proc.shape)
print('  X_test_proc', X_test_proc.shape)

In [ ]:
embeds_clean = compute_methods(X_train_proc, methods=["pca"])

for method, embed in embeds_clean.items():
    fig = plot_3d(embed, 
                  labels=y_train,   # attention : y_train maintenant, pas y
                  method_name=f"{method}", 
                  label_name=f"Gravité {method}")
    fig.show()

---

# Entrainement de modèles

In [24]:
from sklearn.cluster import AgglomerativeClustering, Birch, KMeans, DBSCAN
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score,
    silhouette_score,
    accuracy_score, 
    classification_report, 
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt

### Boucle d'entraînement de méthodes de regroupement
#### Métriques:
- Adjusted Rand Index (ARI): Mesure la similarité entre les vrais regroupements et les regroupements prédits    `(best:1, worst:-0.5)`
- Adjusted Mutual Information (AMI): Similaire à ARI, avec l'information mutuelles des regroupements    `(best:1, worst:0)`
- Homogeneity: La pureté des regroupements comparativement aux vraies classes   `(best:1, worst:0)`
- Silhouette Score: Measure de la séparation entre les regroupements (unsupervised)    `(best:1, worst:-1)`

In [27]:

models = {
    'KMeans': KMeans(n_clusters=c, random_state=seed),
    # 'AgglomerativeClustering': AgglomerativeClustering(n_clusters=c),
    # 'Birch': Birch(n_clusters=c),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
}

results = {}
for name, model in models.items():
    model.fit(X_train_proc)
    if hasattr(model, 'predict'):
        labels_val = model.predict(X_val_proc)
    else:
        labels_val = model.fit_predict(X_val_proc)

    ari = adjusted_rand_score(y_val, labels_val)
    ami = adjusted_mutual_info_score(y_val, labels_val)
    hom = homogeneity_score(y_val, labels_val)
    v = v_measure_score(y_val, labels_val)
    sil = silhouette_score(X_val_proc, labels_val)

    results[name] = {
        'ARI': ari,
        'AMI': ami,
        'Homogeneity': hom,
        'Silhouette': sil,
    }

    print(f'{name}:')
    print(f'  ARI: {ari:.4f}')
    print(f'  AMI: {ami:.4f}')
    print(f'  Homogeneity: {hom:.4f}')
    print(f'  Silhouette: {sil:.4f}')

print('Résumé des performances :')
for name, metrics in results.items():
    print(
        f'{name}: ARI={metrics["ARI"]:.4f}, AMI={metrics["AMI"]:.4f}, Silhouette={metrics["Silhouette"]:.4f}'
    )

KMeans:
  ARI: -0.0179
  AMI: 0.0072
  Homogeneity: 0.0102
  Silhouette: 0.2661
DBSCAN:
  ARI: -0.0334
  AMI: 0.0073
  Homogeneity: 0.0110
  Silhouette: -0.6407
Résumé des performances :
KMeans: ARI=-0.0179, AMI=0.0072, Silhouette=0.2661
DBSCAN: ARI=-0.0334, AMI=0.0073, Silhouette=-0.6407


### Boucle d'entraînement de classifieurs

Nous définissons plusieurs classifieurs scikit-learn, puis comparons leurs performances sur le jeu de validation.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


models = {
    'Dummy (majorite)': DummyClassifier(strategy='most_frequent', random_state=seed),
    'Dummy (stratifie)': DummyClassifier(strategy='stratified', random_state=seed),
    'GaussianNB': GaussianNB(),
    'SVM': LinearSVC(random_state=seed),
    'Decision Tree': DecisionTreeClassifier(random_state=seed, max_depth=10),
    'Random Forest': RandomForestClassifier(random_state=seed)
}

results = {}
trained_models = {}
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    y_val_pred = model.predict(X_val_proc)
    score = accuracy_score(y_val, y_val_pred)
    results[name] = score
    trained_models[name] = model
    print(f'{name}: validation accuracy = {score:.4f}')
    print("Classification report:")
    print(classification_report(y_val, y_val_pred))
    
    # plot confusion matrix
    sns.heatmap(confusion_matrix(y_val, y_val_pred), annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

print('Résumé des performances :')
for name, score in results.items():
    print(f'  {name}: {score:.4f}')

# Déséquilibre de classes
La classe "Grave" ne représente que <1% des données. Du coup les modèles ont tendance à l'ignorer pour maximiser l'accuracy globale. On compare alors trois stratégies pour améliorer la détection des cas graves, toutes appliquées au Random Forest :

1. **Aucune** : baseline, on laisse faire le déséquilibre
2. **`class_weight='balanced'`** : on pénalise plus fortement les erreurs sur les classes rares (sans toucher aux données)
3. **SMOTE** : on génère des exemples synthétiques de la classe minoritaire par interpolation


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, balanced_accuracy_score

# SMOTE une seule fois
smote = SMOTE(random_state=seed)
X_tr_smote, y_tr_smote = smote.fit_resample(X_train_proc, y_train)

strategies_models = {
    'Aucune': {
        'GaussianNB': GaussianNB(),
        'SVM': LinearSVC(random_state=seed, max_iter=2000),
        'Random Forest': RandomForestClassifier(random_state=seed, n_estimators=200, n_jobs=-1),
    },
    'Balanced': {
        'GaussianNB': GaussianNB(priors=[1/3, 1/3, 1/3]),
        'SVM': LinearSVC(random_state=seed, max_iter=2000, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(
            random_state=seed,
            n_estimators=200,
            n_jobs=-1,
            class_weight='balanced'
        ),
    },
    'SMOTE': {
        'GaussianNB': GaussianNB(),
        'SVM': LinearSVC(random_state=seed, max_iter=2000),
        'Random Forest': RandomForestClassifier(random_state=seed, n_estimators=200, n_jobs=-1),
    },
}

comparison = []
trained = {}

for strategie, models_dict in strategies_models.items():

    X_tr = X_tr_smote if strategie == 'SMOTE' else X_train_proc
    y_tr = y_tr_smote if strategie == 'SMOTE' else y_train

    trained[strategie] = {}

    for model_name, model in models_dict.items():

        model.fit(X_tr, y_tr)
        trained[strategie][model_name] = model

        y_pred = model.predict(X_val_proc)

        f1_per_class = f1_score(
            y_val,
            y_pred,
            labels=['Grave', 'Leger', 'Materiel'],
            average=None,
            zero_division=0
        )

        comparison.append({
            'Strategie': strategie,
            'Modele': model_name,
            'Accuracy': accuracy_score(y_val, y_pred),
            'Balanced Acc': balanced_accuracy_score(y_val, y_pred),
            'F1 macro': f1_score(y_val, y_pred, average='macro', zero_division=0),
            'F1 Grave': f1_per_class[0],
        })

comparison_df = pd.DataFrame(comparison)

print("=== Comparaison strategies × modeles ===")
print(comparison_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

pivot = comparison_df.pivot(index='Modele', columns='Strategie', values='F1 macro')

print("\n=== Pivot : F1 macro par modele × strategie ===")
print(pivot.to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

labels = ['Grave', 'Leger', 'Materiel']
model_names = ['GaussianNB', 'SVM', 'Random Forest']

for i, model_name in enumerate(model_names):
    for j, strategie in enumerate(strategies_models.keys()):

        ax = axes[i, j]
        model = trained[strategie][model_name]

        y_pred = model.predict(X_val_proc)

        cm = confusion_matrix(y_val, y_pred, labels=labels)

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=ax,
            xticklabels=labels,
            yticklabels=labels
        )

        ax.set_title(f"{model_name} - {strategie}")

plt.tight_layout()
plt.show()